# 40-1 Exploración de Texto con PySpark

Este notebook demuestra cómo realizar exploración básica de texto usando únicamente funciones nativas de PySpark, sin dependencias externas de NLP.

## Objetivo
Cargar un archivo de texto, tokenizar su contenido, calcular frecuencias de palabras y generar una nube de palabras para visualización.

## Flujo del notebook
1. Instalación de dependencias (solo visualización).
2. Importación de módulos PySpark.
3. Creación de la sesión Spark.
4. Carga del archivo de texto en un DataFrame.
5. Tokenización y normalización con funciones nativas de PySpark.
6. Conteo de frecuencia de palabras.
7. Visualización con nube de palabras (WordCloud).

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa y sin errores.
- Confirma que la ruta del archivo de texto exista en tu entorno.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instalan las librerías necesarias para visualización (`matplotlib`, `wordcloud`) y manejo de datos (`numpy`, `pandas`). No se requiere ninguna librería externa de NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn wordcloud

## 2. Importación de módulos PySpark
Se importan los módulos de PySpark necesarios para la sesión, tipos de datos y funciones de manipulación de DataFrames.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, TimestampType, LongType
from pyspark.sql.types import ArrayType, DoubleType, BooleanType, DecimalType
from pyspark.sql.functions import regexp_extract, split, from_unixtime, col, avg, min, max
from pyspark.sql.functions import grouping_id, window, explode, to_json, from_json
from pyspark.sql.functions import udf, lit, current_timestamp, current_date, date_format

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("exploracion_texto") \
    .getOrCreate()

## 4. Verificación de versiones
Se imprimen las versiones de Spark y Python para confirmar que el entorno está correctamente configurado.


In [ ]:
print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")
spark.sql("select version()").show()

## 5. Importaciones adicionales para procesamiento de texto
Se importan las funciones de PySpark que se usarán para tokenización y normalización: `split`, `lower`, `regexp_replace`, `trim`, `explode`.


In [ ]:
from pyspark.sql.functions import lower, regexp_replace, trim, split, explode, col, size

## 6. Carga del archivo de texto
Se lee el archivo de texto línea por línea en un DataFrame de Spark con una sola columna `linea_texto`.

**Sugerencia:** Verifica que la ruta del archivo exista y ajústala según tu entorno.


In [ ]:
# Cargar el archivo de texto en un DataFrame de Spark
df_texto = spark.read.text("/Users/joseaguilar/Documents/Development/databricks/spark_databricks/programa.txt").toDF("linea_texto")
df_texto.show(5)

## 7. Tokenización y normalización con PySpark nativo
Se reemplaza la pipeline de Spark NLP por funciones nativas de PySpark:
- `lower()` → convierte a minúsculas.
- `regexp_replace()` → elimina caracteres no deseados (puntuación, símbolos).
- `trim()` → elimina espacios sobrantes.
- `split()` → divide el texto en tokens (palabras) por espacios.

El resultado es una columna `palabras` con un array de tokens por línea.


In [ ]:
# Tokenización y normalización usando funciones nativas de PySpark
# 1. Convertir a minúsculas
df_lower = df_texto.withColumn("linea_limpia", lower(col("linea_texto")))

# 2. Eliminar caracteres no deseados (mantener letras, números, acentos, ñ y espacios)
df_clean = df_lower.withColumn(
    "linea_limpia",
    regexp_replace(col("linea_limpia"), r"[^\w\sáéíóúÁÉÍÓÚüÜñÑ¿?¡!]", "")
)

# 3. Reemplazar múltiples espacios por uno solo y hacer trim
df_clean = df_clean.withColumn(
    "linea_limpia",
    trim(regexp_replace(col("linea_limpia"), r"\s+", " "))
)

# 4. Dividir en tokens (palabras)
df_tokens = df_clean.withColumn("palabras", split(col("linea_limpia"), " "))

# Mostrar resultado
df_tokens.select("palabras").show(5, truncate=False)

## 8. Conteo de frecuencia de palabras
Se hace `explode` de la columna de arrays para obtener una palabra por fila, se filtran tokens vacíos y se agrupan para contar la frecuencia de cada palabra.


In [ ]:
# Explode de la columna de listas de palabras para tener una palabra por fila
from pyspark.sql.functions import explode, col
df_palabras = df_tokens.select(explode(col("palabras")).alias("palabra"))
# Filtrar posibles tokens vacíos o muy cortos si aplica
df_palabras = df_palabras.filter(col("palabra") != "")

# Contar frecuencia de cada palabra
df_freq = df_palabras.groupBy("palabra").count().orderBy(col("count").desc())
df_freq.show(10)

## 9. Visualización: Nube de palabras
Se recopilan las 100 palabras más frecuentes en Pandas y se genera una nube de palabras con `WordCloud` y `matplotlib`.


In [ ]:
# Recopilar las palabras más frecuentes en Pandas para visualización
top_words = df_freq.limit(100).toPandas()  # top 100 palabras
palabras = top_words['palabra'].values
frecuencias = top_words['count'].values

# Crear la nube de palabras
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(width=800, height=600, background_color="white", colormap="viridis")
freqs_dict = {palabra: int(freq) for palabra, freq in zip(palabras, frecuencias)}
wc.generate_from_frequencies(freqs_dict)

plt.figure(figsize=(10,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()
